# 04 · Evaluation of every model: paper replication, extensions and fixes

Out-of-sample test quarters 2014Q1-2023Q4, China A-shares with the tradability rules of PAPER_ISSUES A15.
Each table reports what the paper reports, plus the fixes in PAPER_ISSUES.md: one- and two-sided p-values
(C5), raw H-L next to (H-L)/2 (A7), a validation-tuned MSE-Opt benchmark (D2), gross and net results at every
risk aversion (C2, C6), realized frontiers and volatility-matched tests (D3), the warm-start check (D1) and a
Holm adjustment across the E2E-vs-MSE comparisons. Sections whose runs are not finished are skipped.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

from mlm import metrics
from mlm.data import rolling_windows
from mlm.e2e import backtest_meansd, load_sigma
from mlm.linear import GRIDS as LINEAR
from mlm.metrics import drawdown, nw_test, oos_r2, perf
from mlm.mixture import GRID as MIXTURE
from mlm.models import MODELS
from mlm.portfolio import decile_returns, top_portfolio, tuned_kappa
from mlm.train import load_predictions

PANEL_DIR, PRED_DIR = Path("D:/ML_Markowitz_data/panel_cn"), Path("D:/ML_Markowitz_data/preds/cn")
UNRANKED = Path("D:/ML_Markowitz_data/panel_cn_unranked")
RES, FIG = Path("results/cn"), Path("figures/cn")
RES.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
metrics.ANN = json.loads((PANEL_DIR / "meta.json").read_text())["ann"]
windows = rolling_windows("2014-01-01", "2023-12-31")
EXT = ["PCA20-Linear", "PCA20-NN3", "PLS5-NN3", "LDA9-NN3", "SFA20-NN3", "Linear-SVR", "RFF-SVR"] + list(MIXTURE)
have = lambda m: (PRED_DIR / m).exists() and any((PRED_DIR / m).glob("20*Q*.parquet"))
two_stage = [m for m in MODELS + list(LINEAR) + EXT if have(m)]
e2e = [d.name for d in sorted(PRED_DIR.iterdir()) if d.name.startswith(("E2E-", "DA")) and not d.name.startswith(("E2E-tc", "E2E-scratch")) and have(d.name)]
print("two-stage:", two_stage); print("E2E:", e2e)

## Table 2 analogue: decile portfolios, accuracy, and E2E vs its MSE starting point

In [ ]:
rows, tops = {}, {}
for m in two_stage + e2e:
    p = load_predictions(PRED_DIR, m)
    pr, tr = p.groupby("date")["pred"].rank(), p.groupby("date")["target"].rank()
    ic = pd.concat([pr, tr], axis=1).groupby(p["date"]).corr().xs("pred", level=1)["target"]
    dec, tp = decile_returns(p), top_portfolio(p)
    tops[m] = tp["gross"]
    h, l, hl, hlr = (perf(dec[c]) for c in ("High(H)", "Low(L)", "H-L", "H-L raw"))
    rows[m] = {"R2 %": 100 * oos_r2(p.pred, p.target), "IC": ic.mean(), "IC t": nw_test(ic)["t"],
               "H Avg %": h["Avg %"], "H SD %": h["SD %"], "H SR": h["SR"], "L Avg %": l["Avg %"],
               "H-L Avg %": hl["Avg %"], "H-L SR": hl["SR"], "H-L raw Avg %": hlr["Avg %"], "TO %": 100 * tp["to_p"].mean()}
t2 = pd.DataFrame(rows).T
# E2E vs the two-stage model it started from: difference in daily top-decile return
base = lambda m: m.split("-", 1)[1] if m.startswith("E2E-") else m.split("-", 1)[1]
tests = {m: nw_test(tops[m] - tops[base(m)]) for m in e2e if base(m) in tops}
if tests:
    tt = pd.DataFrame(tests).T
    tt["p two-sided (Holm)"] = multipletests(tt["p two-sided"], method="holm")[1]
    t2 = t2.join(tt.add_prefix("vs MSE: "))
t2.to_csv(RES / "table2_all.csv"); t2.round(3)

## Table 3 analogue: transaction costs (NN3)

In [ ]:
nn3, nn3_val = load_predictions(PRED_DIR, "NN3"), load_predictions(PRED_DIR, "NN3", "val")
rows = []
for bps in (0, 10, 20, 30, 40):
    g = bps / 1e4
    kappa, chosen = tuned_kappa(nn3, nn3_val, windows, g) if bps else (None, None)
    runs = {"MSE-Top": top_portfolio(nn3, g), "MSE-Opt": top_portfolio(nn3, g, optimize=True)}
    if bps:
        runs["MSE-Opt (tuned)"] = top_portfolio(nn3, g, optimize=True, kappa=kappa)
    e2e_name = "E2E-NN3" if bps == 0 else f"E2E-tc{bps}-NN3"
    if have(e2e_name):  # E2E signal, same cost-aware construction as MSE-Opt
        runs["E2E"] = top_portfolio(load_predictions(PRED_DIR, e2e_name), g, optimize=bps > 0)
    for k, d in runs.items():
        r = pd.concat([pd.Series({"TCR": bps, "Method": k}), perf(d["net"]).set_axis(["Ann Ret %", "Ann SD %", "Ann SR"]),
                       pd.Series({"TO-P %": 100 * d["to_p"].mean(), "TO-S %": 100 * d["to_s"].mean()}), drawdown(d["net"])])
        if k != "MSE-Top":
            r = pd.concat([r, nw_test(d["net"] - runs["MSE-Opt"]["net"]).add_prefix("vs MSE-Opt: ")]) if k != "MSE-Opt" else r
        rows.append(r)
t3 = pd.DataFrame(rows).set_index(["TCR", "Method"]); t3.to_csv(RES / "table3_all.csv"); t3.round(3)

## Tables 4 and 5 analogue: risk aversion (mean-SD), E2E vs MSE-Opt vs PPP

In [ ]:
ETAS = (0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 7.5, 10.0)
daily = lambda tag: pd.concat([pd.read_parquet(f) for f in sorted((PRED_DIR / f"{tag}-NN3").glob("*_daily.parquet"))]).set_index("date").sort_index() \
    if (PRED_DIR / f"{tag}-NN3").exists() else None
sig = load_sigma(UNRANKED, nn3)
rows, series = [], {}
for eta in ETAS:
    for g, gamma in (("gross", 0.0), ("net", 0.002)):
        cands = {"MSE-Opt": backtest_meansd(nn3, sig, eta, gamma), "E2E": daily(f"MSD-{g}-eta{eta:g}"), "PPP": daily(f"PPP-{g}-eta{eta:g}")}
        for k, d in cands.items():
            if d is None:
                continue
            r = d["net"] if gamma else d["gross"]
            series[(eta, g, k)] = r
            st = perf(r)
            rows.append({"eta": eta, "type": g, "method": k, "Daily ret %": 100 * r.mean(), "Daily vol %": 100 * r.std(),
                         "Utility %": 100 * (r.mean() - eta * r.std()), "Ann Ret %": st["Avg %"], "Ann Vol %": st["SD %"], "SR": st["SR"]})
t4 = pd.DataFrame(rows).pivot_table(index=["eta", "method"], columns="type").round(3) if rows else None
t4

In [ ]:
# Table 5: pooled tests across eta. Volatility-matched (scaled to 20% a year) return, and realized utility.
out = {}
for g in ("gross", "net"):
    for bench in ("MSE-Opt", "PPP"):
        vm, ut = [], []
        for eta in ETAS:
            a, b = series.get((eta, g, "E2E")), series.get((eta, g, bench))
            if a is None or b is None:
                continue
            scale = lambda r: r * 0.20 / (r.std() * np.sqrt(metrics.ANN))
            vm.append(scale(a) - scale(b))
            ut.append((a - eta * a.std()) - (b - eta * b.std()))
        if vm:
            out[(g, f"E2E vs {bench}")] = pd.concat([nw_test(pd.concat(vm)).add_prefix("vol-matched "), nw_test(pd.concat(ut)).add_prefix("utility ")])
pd.DataFrame(out).T.round(4) if out else "Section E of notebook 03 has not been run yet"

In [ ]:
# realized efficient frontiers (Figure 2)
if series:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    colors = {"E2E": "#2a78d6", "MSE-Opt": "#eb6834", "PPP": "#1baf7a"}
    for k, c in colors.items():
        for g, ls in (("gross", "--"), ("net", "-")):
            pts = [(series[(e, g, k)].std() * np.sqrt(metrics.ANN), series[(e, g, k)].mean() * metrics.ANN) for e in ETAS if (e, g, k) in series]
            if pts:
                x, y = zip(*pts)
                ax.plot(x, y, ls, marker="o", ms=3, color=c, label=f"{k} ({g})")
    ax.set_xlabel("annualized standard deviation"); ax.set_ylabel("annualized market-adjusted return")
    ax.legend(fontsize=8, frameon=False); ax.grid(alpha=0.3)
    fig.savefig(FIG / "frontier.png", dpi=150, bbox_inches="tight")

## Warm-start check (PAPER_ISSUES D1): E2E from scratch vs warm-started, same windows

In [ ]:
if have("E2E-scratch-NN3") and have("E2E-NN3"):
    s = top_portfolio(load_predictions(PRED_DIR, "E2E-scratch-NN3"))["gross"]
    w = tops["E2E-NN3"].loc[s.index]
    pd.DataFrame({"from scratch": perf(s), "warm start": perf(w)}).T.join(pd.DataFrame([nw_test(w - s)], index=["warm start"]))